In [1]:
%%writefile session1_seed.py
import argparse
import json
import os
import random
from datetime import datetime, timedelta

import pandas as pd

SEED = 26
CITIES = ["Lahore", "Karachi", "Islamabad", "Rawalpindi", "Faisalabad", "Multan", "Peshawar", "Quetta", "Sialkot", "Hyderabad"]
CITY_WEIGHTS = [22, 25, 12, 8, 9, 7, 6, 3, 4, 4]
FIRST = ["Ali", "Ahmed", "Fatima", "Ayesha", "Hassan", "Zainab", "Usman", "Maryam", "Bilal", "Hira",
         "Omar", "Sana", "Hamza", "Amna", "Saad", "Iqra", "Talha", "Noor", "Imran", "Mahnoor"]
LAST = ["Khan", "Malik", "Butt", "Sheikh", "Chaudhry", "Qureshi", "Raza", "Siddiqui", "Mirza", "Abbasi"]
SEGMENTS = ["Consumer", "Corporate", "Small Business"]
CATEGORIES = {"Electronics": (2500, 90000), "Grocery": (300, 6000), "Fashion": (900, 15000),
              "Home": (1500, 40000), "Beauty": (400, 8000), "Books": (300, 3500)}
STATUSES = ["delivered", "shipped", "pending", "cancelled", "returned"]
STATUS_WEIGHTS = [55, 20, 12, 8, 5]
PAYMENTS = ["card", "cash_on_delivery", "bank_transfer", "wallet"]

START = datetime(2025, 1, 1)
END = datetime(2026, 10, 24, 23, 59)


def make_customers(rng, n=2000):
    rows, movers = [], {}
    for cid in range(1, n + 1):
        signup = START - timedelta(days=rng.randint(0, 900))
        rows.append({
            "customer_id": cid,
            "name": f"{rng.choice(FIRST)} {rng.choice(LAST)}",
            "city": rng.choices(CITIES, CITY_WEIGHTS)[0],
            "segment": rng.choices(SEGMENTS, [70, 15, 15])[0],
            "signup_date": signup.date(),
        })
        if rng.random() < 0.05:
            old = rng.choices([c for c in CITIES if c != rows[-1]["city"]])[0]
            movers[cid] = (old, datetime(2026, 1, 1) + timedelta(days=rng.randint(0, 240)))
    return pd.DataFrame(rows), movers


def make_orders(rng, customers, n=200_000):
    span = int((END - START).total_seconds())
    cust_ids = customers["customer_id"].tolist()
    weights = [min(rng.paretovariate(1.5), 15) for _ in cust_ids]
    picks = rng.choices(cust_ids, weights, k=n)
    rows = []
    for oid, cid in enumerate(picks, start=100001):
        order_ts = START + timedelta(seconds=rng.randint(0, span))
        cat = rng.choice(list(CATEGORIES))
        lo, hi = CATEGORIES[cat]
        qty = rng.choices([1, 2, 3, 4, 5], [60, 22, 10, 5, 3])[0]
        amount = round(rng.uniform(lo, hi) * qty, 2)
        status = rng.choices(STATUSES, STATUS_WEIGHTS)[0]
        days_after = {"pending": 0, "shipped": rng.randint(1, 3), "delivered": rng.randint(2, 8),
                      "cancelled": rng.randint(0, 2), "returned": rng.randint(10, 25)}[status]
        updated = min(order_ts + timedelta(days=days_after, minutes=rng.randint(0, 600)), END)
        rows.append({
            "order_id": oid,
            "customer_id": cid,
            "order_date": order_ts.date(),
            "order_ts": order_ts,
            "category": cat,
            "quantity": qty,
            "amount": amount,
            "status": status,
            "payment_method": rng.choice(PAYMENTS),
            "updated_at": updated,
        })
    return pd.DataFrame(rows).sort_values("order_ts").reset_index(drop=True)


def make_documents(rng, orders, customers, movers, n=5000):
    cust = customers.set_index("customer_id")
    sample = orders.sample(n=n, random_state=SEED).sort_values("order_ts")
    docs = []
    for r in sample.itertuples(index=False):
        doc = {
            "order_id": int(r.order_id),
            "order_date": r.order_ts.isoformat(),
            "status": r.status,
            "amount": float(r.amount),
            "items": [{"category": r.category, "quantity": int(r.quantity)}],
            "payment": {"method": r.payment_method},
            "updated_at": r.updated_at.isoformat(),
        }
        if rng.random() < 0.7:
            c = cust.loc[r.customer_id]
            city = c["city"]
            if r.customer_id in movers and r.order_ts < movers[r.customer_id][1]:
                city = movers[r.customer_id][0]
            doc["customer"] = {"id": int(r.customer_id), "name": c["name"], "city": city}
            if rng.random() < 0.3:
                doc["customer"]["segment"] = c["segment"]
        else:
            doc["customer_id"] = int(r.customer_id)
        if rng.random() < 0.2:
            doc["coupon"] = rng.choice(["WELCOME10", "ELEVEN11", "EID25", "FREESHIP"])
        if rng.random() < 0.1:
            doc["delivery_notes"] = rng.choice(["Call before arriving", "Leave with guard", "Office hours only"])
        if rng.random() < 0.15:
            doc["payment"]["installments"] = rng.choice([3, 6, 12])
        docs.append(doc)
    return docs


def write_files(orders, customers, docs, out="data"):
    os.makedirs(out, exist_ok=True)
    orders.drop(columns=["order_ts"]).to_csv(f"{out}/orders.csv", index=False)
    customers.to_csv(f"{out}/customers.csv", index=False)
    with open(f"{out}/orders.json", "w") as f:
        json.dump(docs, f, indent=1)
    print(f"Wrote {out}/orders.csv ({len(orders):,} rows), {out}/customers.csv, {out}/orders.json ({len(docs):,} docs)")


def load_postgres(url, orders, customers, ro_password):
    from sqlalchemy import create_engine, text

    url = url.replace("postgres://", "postgresql://", 1).replace("postgresql://", "postgresql+psycopg2://", 1)
    engine = create_engine(url)
    with engine.begin() as con:
        con.execute(text("DROP TABLE IF EXISTS orders"))
        con.execute(text("DROP TABLE IF EXISTS customers"))
        con.execute(text("""
            CREATE TABLE customers (
                customer_id  INTEGER PRIMARY KEY,
                name         TEXT NOT NULL,
                city         TEXT NOT NULL,
                segment      TEXT NOT NULL,
                signup_date  DATE NOT NULL
            )"""))
        con.execute(text("""
            CREATE TABLE orders (
                order_id        INTEGER PRIMARY KEY,
                customer_id     INTEGER NOT NULL REFERENCES customers(customer_id),
                order_date      DATE NOT NULL,
                category        TEXT NOT NULL,
                quantity        INTEGER NOT NULL,
                amount          NUMERIC(12, 2) NOT NULL,
                status          TEXT NOT NULL,
                payment_method  TEXT NOT NULL,
                updated_at      TIMESTAMP NOT NULL
            )"""))
    customers.to_sql("customers", engine, if_exists="append", index=False, method="multi", chunksize=1000)
    print("Loading 200k orders into Postgres, about 1 to 3 minutes...")
    orders.drop(columns=["order_ts"]).to_sql("orders", engine, if_exists="append", index=False,
                                             method="multi", chunksize=5000)
    with engine.begin() as con:
        con.execute(text("CREATE INDEX IF NOT EXISTS idx_orders_updated_at ON orders(updated_at)"))
        con.execute(text("CREATE INDEX IF NOT EXISTS idx_orders_order_date ON orders(order_date)"))
        n = con.execute(text("SELECT COUNT(*) FROM orders")).scalar()
        print(f"Postgres: customers={len(customers):,}, orders={n:,}")
        if ro_password:
            db = con.execute(text("SELECT current_database()")).scalar()
            exists = con.execute(text("SELECT 1 FROM pg_roles WHERE rolname = 'student_ro'")).scalar()
            verb = "ALTER" if exists else "CREATE"
            pw = ro_password.replace("'", "''")
            con.execute(text(f"{verb} ROLE student_ro WITH LOGIN PASSWORD '{pw}'"))
            con.execute(text(f'GRANT CONNECT ON DATABASE "{db}" TO student_ro'))
            con.execute(text("GRANT USAGE ON SCHEMA public TO student_ro"))
            con.execute(text("GRANT SELECT ON customers, orders TO student_ro"))
            print("Read-only role student_ro is ready.")


def load_mongo(url, docs):
    from pymongo import MongoClient

    client = MongoClient(url, serverSelectionTimeoutMS=15000)
    col = client["shop"]["orders"]
    col.drop()
    col.insert_many([dict(d) for d in docs])
    col.create_index("status")
    col.create_index("updated_at")
    print(f"MongoDB: shop.orders has {col.count_documents({}):,} documents "
          f"({col.count_documents({'status': 'shipped'}):,} shipped)")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--pg", default=os.getenv("PG_URL"))
    p.add_argument("--mongo", default=os.getenv("MONGO_URL"))
    p.add_argument("--readonly-password", default=os.getenv("RO_PASSWORD"))
    p.add_argument("--orders", type=int, default=200_000)
    p.add_argument("--docs", type=int, default=5000)
    p.add_argument("--out", default="data")
    p.add_argument("--skip-pg", action="store_true")
    p.add_argument("--skip-mongo", action="store_true")
    p.add_argument("--files-only", action="store_true")
    a = p.parse_args()

    rng = random.Random(SEED)
    customers, movers = make_customers(rng)
    orders = make_orders(rng, customers, a.orders)
    docs = make_documents(rng, orders, customers, movers, a.docs)
    write_files(orders, customers, docs, a.out)

    if not (a.files_only or a.skip_pg):
        if not a.pg:
            raise SystemExit("Pass --pg (or set PG_URL), or use --skip-pg")
        load_postgres(a.pg, orders, customers, a.readonly_password)
    if not (a.files_only or a.skip_mongo):
        if not a.mongo:
            raise SystemExit("Pass --mongo (or set MONGO_URL), or use --skip-mongo")
        load_mongo(a.mongo, docs)


if __name__ == "__main__":
    main()

Writing session1_seed.py


In [2]:
!pip -q install psycopg2_binary pymongo sqlalchemy pandas pyarrow requests

import os , json , requests , pandas as pd
from datetime import date
from google.colab import userdata
from sqlalchemy import create_engine
from pymongo import MongoClient

PG_URL = userdata.get("PG_URL").replace("postgres://", "postgresql://", 1)
PG_URL = PG_URL.replace("postgresql://", "postgresql+psycopg2://", 1)
MONGO_URL = userdata.get("MONGO_URL")

os.makedirs("raw" , exist_ok = True)
TODAY = date.today().isoformat()
engine = create_engine(PG_URL)


In [3]:
!python session1_seed.py --pg "$PG_URL" --mongo "$MONGO_URL"

Wrote data/orders.csv (200,000 rows), data/customers.csv, data/orders.json (5,000 docs)
Loading 200k orders into Postgres, about 1 to 3 minutes...
Postgres: customers=2,000, orders=200,000
Traceback (most recent call last):
  File "/content/session1_seed.py", line 209, in <module>
    main()
    ~~~~^^
  File "/content/session1_seed.py", line 205, in main
    load_mongo(a.mongo, docs)
    ~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/content/session1_seed.py", line 171, in load_mongo
    col.drop()
    ~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/pymongo/synchronous/collection.py", line 1471, in drop
    dbo.drop_collection(
    ~~~~~~~~~~~~~~~~~~~^
        self._name, session=session, comment=comment, encrypted_fields=encrypted_fields
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/pymongo/_csot.py", line 126, in csot_wrapper
    return func(self, *args, **kwargs)
  File "/usr/local/

In [4]:
customers = pd.read_sql("SELECT * FROM customers", engine)
customers.to_parquet(f"raw/customers_{TODAY}.parquet", index=False)

chunks = []
for chunk in pd.read_sql("SELECT * FROM orders ORDER BY order_id", engine, chunksize=50_000):
    chunks.append(chunk)
    print("chunk rows:", len(chunk))
orders_pg = pd.concat(chunks, ignore_index=True)
orders_pg.to_parquet(f"raw/orders_pg_{TODAY}.parquet", index=False)

print("customers:", len(customers), "| orders:", len(orders_pg))

chunk rows: 50000
chunk rows: 50000
chunk rows: 50000
chunk rows: 50000
customers: 2000 | orders: 200000


In [5]:
import json
import os

# Fallback: Load the local JSON backup written by the seed script
local_json_path = "data/orders.json"

if os.path.exists(local_json_path):
    with open(local_json_path, "r") as f:
        docs = json.load(f)

    # Ensure Raw folder exists
    os.makedirs("raw", exist_ok=True)

    # Save the documents in your raw folder to match your expected workflow
    with open(f"raw/orders_mongo_{TODAY}.json", "w") as f:
        json.dump(docs, f, default=str)

    print(f"Successfully loaded {len(docs)} documents from local backup file: {local_json_path}")
else:
    print(f"Error: {local_json_path} not found. Please verify the file path.")

Successfully loaded 5000 documents from local backup file: data/orders.json


In [6]:
import requests
import json
import os
from datetime import date

TODAY = date.today().isoformat()
CITY_COORDS = {"Lahore": (31.55, 74.34), "Karachi": (24.86, 67.01), "Islamabad": (33.69, 73.05)}

# Ensure the target raw directory exists
os.makedirs("raw", exist_ok=True)

for city, (lat, lon) in CITY_COORDS.items():
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={lat}&longitude={lon}&start_date=2025-01-01&end_date=2026-10-24"
           "&daily=temperature_2m_max,precipitation_sum&timezone=Asia/Karachi")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(f"raw/weather_{city.lower()}_{TODAY}.json", "w") as f:
        json.dump(resp.json(), f)
    print(city, "days:", len(resp.json()["daily"]["time"]))


Lahore days: 662
Karachi days: 662
Islamabad days: 662


In [7]:
from sqlalchemy import text
import pandas as  pd

def extract_orders(last_run):
    q = text("SELECT * FROM orders WHERE updated_at > :lr ORDER BY updated_at")
    return pd.read_sql(q, engine, params={"lr": last_run})

inc = extract_orders("2026-10-01")
print("Rows since 2026-10-01:", len(inc))

Rows since 2026-10-01: 8593


In [8]:
for f in sorted(os.listdir("raw")):print(f)

customers_2026-11-04.parquet
orders_mongo_2026-11-04.json
orders_pg_2026-11-04.parquet
weather_islamabad_2026-11-04.json
weather_karachi_2026-11-04.json
weather_lahore_2026-11-04.json


In [9]:
with open(f"raw/orders_mongo_{TODAY}.json") as f:
    docs = json.load(f)

m = pd.json_normalize(docs)
print(m.columns.tolist())

app = pd.DataFrame({
    "order_id":       m["order_id"].astype(int),
    "customer_id":    m["customer.id"].fillna(m["customer_id"]).astype(int),
    "order_date":     pd.to_datetime(m["order_date"]).dt.normalize(),
    "category":       m["items"].str[0].str["category"],
    "quantity":       m["items"].str[0].str["quantity"].astype(int),
    "amount":         m["amount"].astype(float),
    "status":         m["status"],
    "payment_method": m["payment.method"],
    "updated_at":     pd.to_datetime(m["updated_at"]),
    "app_city":       m["customer.city"],      # keep for SCD Type 2 in Part 3
})
app["source"] = "app"

web = orders_pg.copy()
web["order_date"] = pd.to_datetime(web["order_date"])
web["updated_at"] = pd.to_datetime(web["updated_at"])
web["amount"] = web["amount"].astype(float)
web["app_city"] = None
web["source"] = "web"

['order_id', 'order_date', 'status', 'amount', 'items', 'updated_at', 'payment.method', 'payment.installments', 'customer.id', 'customer.name', 'customer.city', 'customer_id', 'coupon', 'delivery_notes', 'customer.segment']


In [10]:
both = set(web["order_id"]) & set(app["order_id"])
print("order_ids in both sources:", len(both))

combined = pd.concat([web, app], ignore_index=True)
combined["in_app"] = combined["order_id"].isin(both) | (combined["source"] == "app")

# Rule: keep one row per order_id, the one with the latest updated_at;
# on a tie keep the web row (PostgreSQL is the system of record).
combined["_prio"] = (combined["source"] == "web").astype(int)
combined = (combined.sort_values(["order_id", "updated_at", "_prio"])
                    .drop_duplicates("order_id", keep="last")
                    .drop(columns="_prio"))

# don't lose the app city when the web row wins
city_map = app.set_index("order_id")["app_city"]
combined["app_city"] = combined["app_city"].fillna(combined["order_id"].map(city_map))
orders = combined.reset_index(drop=True)
print("clean orders:", len(orders))

order_ids in both sources: 5000
clean orders: 200000


In [11]:
valid_status = {"delivered", "shipped", "pending", "cancelled", "returned"}
checks = {
    "no null order_id":            orders["order_id"].notna().all(),
    "order_id unique":             orders["order_id"].is_unique,
    "amount > 0":                  (orders["amount"] > 0).all(),
    "customer_id exists":          orders["customer_id"].isin(customers["customer_id"]).all(),
    "no future order_date":        (orders["order_date"] <= pd.Timestamp("2026-10-24")).all(),
    "status in 5 known values":    orders["status"].isin(valid_status).all(),
    "quantity >= 1":               (orders["quantity"] >= 1).all(),
}
qc = pd.DataFrame({"check": checks.keys(), "result": ["PASS" if v else "FAIL" for v in checks.values()]})
qc

,check,result
0,no null order_id,PASS
1,order_id unique,PASS
2,amount > 0,PASS
3,customer_id exists,PASS
4,no future order_date,PASS
5,status in 5 known values,PASS
6,quantity >= 1,PASS


In [12]:
rows = []
for city in CITY_COORDS:
    with open(f"raw/weather_{city.lower()}_{TODAY}.json") as f:
        d = json.load(f)["daily"]
    rows.append(pd.DataFrame({"city": city, "date": pd.to_datetime(d["time"]),
                              "max_temp": d["temperature_2m_max"], "rain_mm": d["precipitation_sum"]}))
weather = pd.concat(rows, ignore_index=True)

orders = orders.merge(customers[["customer_id", "city"]], on="customer_id", how="left")
orders = orders.merge(weather, left_on=["city", "order_date"], right_on=["city", "date"], how="left").drop(columns="date")

print(orders.shape, "| orders with weather:", orders["max_temp"].notna().sum())
orders.to_parquet("clean_orders.parquet", index=False)

(200000, 15) | orders with weather: 117402


In [13]:
# ---- dim_date: one row per day, 2025-01-01 to 2026-12-31 ----
dim_date = pd.DataFrame({"date": pd.date_range("2025-01-01", "2026-12-31")})
dim_date["date_key"]    = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"]        = dim_date["date"].dt.year
dim_date["quarter"]     = dim_date["date"].dt.quarter
dim_date["month"]       = dim_date["date"].dt.month
dim_date["month_name"]  = dim_date["date"].dt.month_name()
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"]  = dim_date["date"].dt.dayofweek >= 5
print("dim_date rows:", len(dim_date))   # 730

# ---- dim_category ----
cats = sorted(orders["category"].unique())
dim_category = pd.DataFrame({"category_key": range(1, len(cats) + 1), "category": cats})
dim_category

dim_date rows: 730


,category_key,category
0,1,Beauty
1,2,Books
2,3,Electronics
3,4,Fashion
4,5,Grocery
5,6,Home


In [14]:

FAR_FUTURE = pd.Timestamp("2099-12-31")
known = orders.loc[orders["app_city"].notna(), ["customer_id", "order_date", "app_city"]].copy()
known["current_city"] = known["customer_id"].map(customers.set_index("customer_id")["city"])


movers = {}
for cid, g in known.groupby("customer_id"):
    cur = g["current_city"].iloc[0]
    old = g[g["app_city"] != cur]
    if old.empty:
        continue
    new = g[g["app_city"] == cur]

    new_start = new["order_date"].min() if len(new) else old["order_date"].max() + pd.Timedelta(days=1)
    movers[cid] = (old["app_city"].iloc[0], new_start)
print("customers who moved:", len(movers))

rows, key = [], 1
for r in customers.itertuples():
    signup = pd.Timestamp(r.signup_date)
    if r.customer_id in movers:
        old_city, new_start = movers[r.customer_id]
        rows.append((key, r.customer_id, r.name, old_city, r.segment, signup, new_start - pd.Timedelta(days=1), False)); key += 1
        rows.append((key, r.customer_id, r.name, r.city,   r.segment, new_start, FAR_FUTURE, True));                  key += 1
    else:
        rows.append((key, r.customer_id, r.name, r.city, r.segment, signup, FAR_FUTURE, True)); key += 1

dim_customer = pd.DataFrame(rows, columns=["customer_key", "customer_id", "name", "city", "segment",
                                           "valid_from", "valid_to", "is_current"])
print("dim_customer rows:", len(dim_customer), "| current rows:", dim_customer["is_current"].sum())

customers who moved: 57
dim_customer rows: 2057 | current rows: 2000


In [15]:

f = orders.merge(dim_customer[["customer_key", "customer_id", "city", "valid_from", "valid_to"]], on="customer_id", how="left", suffixes=("_cur", ""))
f = f[(f["order_date"] >= f["valid_from"]) & (f["order_date"] <= f["valid_to"])].copy()
assert len(f) == len(orders) and f["order_id"].is_unique, "every order must match exactly one customer row"


f = f.drop(columns=["max_temp", "rain_mm"]).merge(weather, left_on=["city", "order_date"], right_on=["city", "date"], how="left").drop(columns=["date", "city"])
f["date_key"] = f["order_date"].dt.strftime("%Y%m%d").astype(int)
f = f.merge(dim_category, on="category", how="left")

fact_orders = f[["order_id", "customer_key", "customer_id", "date_key", "order_date", "category_key", "quantity",
                 "amount", "status", "payment_method", "source", "in_app", "updated_at", "max_temp", "rain_mm"]].copy()
fact_orders = fact_orders.sort_values("order_id").reset_index(drop=True)
print(fact_orders.shape)
fact_orders.head()

(200000, 15)


,order_id,customer_key,customer_id,date_key,order_date,category_key,quantity,amount,status,payment_method,source,in_app,updated_at,max_temp,rain_mm
0,100001,268,261,20260901,2026-09-01,3,1,77635.24,delivered,bank_transfer,web,False,2026-09-07 14:58:52,NaN,NaN
1,100002,102,101,20250228,2025-02-28,5,1,5562.63,delivered,card,web,False,2025-03-05 07:26:15,NaN,NaN
2,100003,802,777,20260708,2026-07-08,2,1,2055.15,shipped,card,web,False,2026-07-09 23:35:51,NaN,NaN
3,100004,1549,1509,20250222,2025-02-22,1,1,7414.43,delivered,wallet,web,False,2025-03-01 21:29:06,21.9,0.0
4,100005,165,162,20260112,2026-01-12,2,4,6784.53,delivered,wallet,web,False,2026-01-14 02:59:49,NaN,NaN


In [16]:

mover_id = dim_customer.groupby("customer_id").size().loc[lambda s: s == 2].index[0]
display(dim_customer[dim_customer["customer_id"] == mover_id])
display(fact_orders[(fact_orders["customer_id"] == mover_id) &
                    (fact_orders["order_date"].between("2026-02-01", "2026-02-28"))]
        [["order_id", "order_date", "customer_key"]])

,customer_key,customer_id,name,city,segment,valid_from,valid_to,is_current
4,5,5,Hamza Qureshi,Quetta,Consumer,2022-11-10,2026-04-14,False
5,6,5,Hamza Qureshi,Karachi,Consumer,2026-04-15,2099-12-31,True


,order_id,order_date,customer_key
7195,107196,2026-02-24,5
14032,114033,2026-02-26,5
37038,137039,2026-02-10,5
71162,171163,2026-02-12,5
95592,195593,2026-02-09,5
108758,208759,2026-02-08,5
154612,254613,2026-02-21,5
198133,298134,2026-02-10,5


In [17]:

import shutil
os.makedirs("export", exist_ok=True)
kw = dict(index=False, coerce_timestamps="ms", allow_truncated_timestamps=True)

dd = dim_date.copy();      dd["date"] = dd["date"].dt.date
dc = dim_customer.copy();  dc["valid_from"] = dc["valid_from"].dt.date; dc["valid_to"] = dc["valid_to"].dt.date
fo = fact_orders.copy();   fo["order_date"] = fo["order_date"].dt.date

dd.to_parquet("export/dim_date.parquet", **kw)
dc.to_parquet("export/dim_customer.parquet", **kw)
dim_category.to_parquet("export/dim_category.parquet", **kw)

fo[fo["updated_at"] <= "2026-10-20"].to_parquet("export/fact_orders_initial.parquet", **kw)
fo[fo["updated_at"] >  "2026-10-20"].to_parquet("export/fact_orders_incremental.parquet", **kw)

print("initial rows:", (fo["updated_at"] <= "2026-10-20").sum(), "| incremental rows:", (fo["updated_at"] > "2026-10-20").sum())
shutil.make_archive("export_for_databricks", "zip", "export")
from google.colab import files
files.download("export_for_databricks.zip")

initial rows: 197153 | incremental rows: 2847


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>